# Tarea 3: Heart Failure Prediction: Validación, Hiperparámetros y Comparación de Modelos

MDS7104: Aprendizaje de Maquinas - Otoño 2026

---

### Cuerpo Docente:

- Profesor: Francisco Vásquez L.
- Auxiliares: Álvaro Márquez y Diego Olguín Wende
- Ayudantes: Javiera Yañez y Tamara Carrasco


### Estudiante

- Felipe Muñoz M.

In [ ]:
!uv add numpy requests pandas matplotlib scipy scikit-learn ruff pre-commit

Resolved 63 packages in 20ms
Audited 58 packages in 16ms


In [3]:
!uv add ipykernel ipython matplotlib-inline

Resolved 130 packages in 24ms
Audited 127 packages in 26ms


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, ConfusionMatrixDisplay
)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [37]:
# %matplotlib inline

# b) Particionamiento y exploración

Se procede a importar la base de datos

In [2]:
# Se abre el Dataframe
df = pd.read_csv("/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/tareas/tarea3/heart.csv")
df.head(5)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


Se realiza la partición de los datos, donde el 80% va al Train, y el 20% va al Test usando stratify

In [27]:
# Se convierten las variables categóricas en variables dummy
df_dummies = pd.get_dummies(df, drop_first=True)

# Se fijan las variables predictoras (X) y la variable objetivo (y)
X = df_dummies.drop(columns="HeartDisease").values
y = df_dummies["HeartDisease"].values
feature_names = df_dummies.drop(columns="HeartDisease").columns.tolist()

# Se fija la semilla
SEED = 7

# Se realiza el Split entre Train y Test
X_train_80, X_test_20, y_train_80, y_test_20 = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=SEED, 
    stratify=y
)

Se verifican la proporción de clases y estadísticas descriptivas de HeartDiseare, y la Media y Desviación estandar de Cholesterol y Age. 

In [4]:
# Proporción de clases en train y test de HeartDisease
print("Proporción de clases en Train:")
print(pd.Series(y_train_80).value_counts(normalize=True))
print("\nProporción de clases en Test:")
print(pd.Series(y_test_20).value_counts(normalize=True))

Proporción de clases en Train:
1    0.553134
0    0.446866
Name: proportion, dtype: float64

Proporción de clases en Test:
1    0.554348
0    0.445652
Name: proportion, dtype: float64


In [5]:
X_train_80

array([[39, 138, 220, ..., False, True, False],
       [51, 128, 0, ..., True, True, False],
       [64, 150, 193, ..., True, True, False],
       ...,
       [45, 110, 264, ..., False, True, False],
       [60, 132, 218, ..., True, False, False],
       [57, 150, 255, ..., True, True, False]],
      shape=(734, 15), dtype=object)

In [6]:
# Se calcula la Media y Desviación Estándar de cada característica en el conjunto de entrenamiento y Test
posición_característica = [0, 4]

print("Conjunto de Entrenamiento:")
for i in posición_característica:
    print(f"Característica {feature_names[i]} - Media: {np.mean(X_train_80[:, i]):.2f}, Desviación Estándar: {np.std(X_train_80[:, i]):.2f}")

print("\nConjunto de Test:")
for i in posición_característica:
    print(f"Característica {feature_names[i]} - Media: {np.mean(X_test_20[:, i]):.2f}, Desviación Estándar: {np.std(X_test_20[:, i]):.2f}")

Conjunto de Entrenamiento:
Característica Age - Media: 53.51, Desviación Estándar: 9.27
Característica MaxHR - Media: 137.42, Desviación Estándar: 24.99

Conjunto de Test:
Característica Age - Media: 53.53, Desviación Estándar: 10.02
Característica MaxHR - Media: 134.37, Desviación Estándar: 27.06


In [7]:
# Se realiza un histograma de la caracteristica "Cholesterol" de Train y Test
fig = make_subplots(rows=2, cols=1, subplot_titles=("Histograma de Cholesterol en Train", "Histograma de Cholesterol en Test"))

fig.add_trace(go.Histogram(x=X_train_80[:, 4], name="Train"), row=1, col=1)
fig.add_trace(go.Histogram(x=X_test_20[:, 4], name="Test", marker_color="red"), row=2, col=1)
fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig.update_yaxes(title_text="Frecuencia", row=2, col=1)

fig.update_layout(height=600, showlegend=False)
fig.show()

# C) Regresión Logísitca baseline

Se ajusta el modelo usando Regresión Lógistica con hiperparámetros por defecto y se realizan las predicciones

In [8]:
# Train del modelo de Regresión Logística
lr_80 = LogisticRegression(random_state=SEED, max_iter=1000)
lr_80.fit(X_train_80, y_train_80)

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",7
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multic

In [9]:
# Predicciones
y_pred_train_80 = lr_80.predict(X_train_80)
y_pred_test_20  = lr_80.predict(X_test_20)

Luego, se crea una función auxiliar para calcular las métricas en train y test: Acc, Prec, Recall, Spec, F1, FPR, FNR. 

In [10]:
def metricas_M(y_true, y_pred, nombre=""):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc   = accuracy_score(y_true, y_pred)
    prec  = precision_score(y_true, y_pred, zero_division=0)
    rec   = recall_score(y_true, y_pred, zero_division=0)      # TPR
    spec  = tn / (tn + fp)                                      # TNR
    f1    = f1_score(y_true, y_pred, zero_division=0)
    fpr   = fp / (fp + tn)
    fnr   = fn / (fn + tp)
    print(f"\n── Métricas {nombre} ──────────────────────────")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {prec:.4f}")
    print(f"  Recall (TPR): {rec:.4f}")
    print(f"  Specificity : {spec:.4f}")
    print(f"  F1-Score    : {f1:.4f}")
    print(f"  FPR         : {fpr:.4f}")
    print(f"  FNR         : {fnr:.4f}")
    return dict(Accuracy=acc, Precision=prec, Recall=rec,
                Specificity=spec, F1=f1, FPR=fpr, FNR=fnr)

metrics_train = metricas_M(y_train_80, y_pred_train_80, "TRAIN")
metrics_test  = metricas_M(y_test_20,  y_pred_test_20,  "TEST")

# Tabla comparativa Train vs Test
tabla = pd.DataFrame({"Train": metrics_train, "Test": metrics_test}).T
print("\n── Tabla comparativa ─────────────────────────")
print(tabla.round(4).to_string())


── Métricas TRAIN ──────────────────────────
  Accuracy    : 0.8733
  Precision   : 0.8735
  Recall (TPR): 0.9015
  Specificity : 0.8384
  F1-Score    : 0.8873
  FPR         : 0.1616
  FNR         : 0.0985

── Métricas TEST ──────────────────────────
  Accuracy    : 0.8478
  Precision   : 0.8558
  Recall (TPR): 0.8725
  Specificity : 0.8171
  F1-Score    : 0.8641
  FPR         : 0.1829
  FNR         : 0.1275

── Tabla comparativa ─────────────────────────
       Accuracy  Precision  Recall  Specificity      F1     FPR     FNR
Train    0.8733     0.8735  0.9015       0.8384  0.8873  0.1616  0.0985
Test     0.8478     0.8558  0.8725       0.8171  0.8641  0.1829  0.1275


Se reporta la matriz de confusión en train y test

In [11]:
# Matrices de confusión (Plotly)
def plot_cm_plotly(y_true, y_pred, titulo):
    if titulo == "Matriz de Confusión – Train":
        color = "Blues"
    else:
        color = "Reds"
    cm = confusion_matrix(y_true, y_pred)
    labels = ["No Enfermedad (0)", "Enfermedad (1)"]
    fig = go.Figure(go.Heatmap(
        z=cm[::-1],                          # invertir para TN arriba-izquierda
        x=labels, y=labels[::-1],
        colorscale=color,
        text=cm[::-1], texttemplate="%{text}",
        showscale=False,
    ))
    fig.update_layout(
        title=titulo,
        xaxis_title="Predicho",
        yaxis_title="Real",
        width=420, height=380
    )
    fig.show()

plot_cm_plotly(y_train_80, y_pred_train_80, "Matriz de Confusión – Train")
plot_cm_plotly(y_test_20,  y_pred_test_20,  "Matriz de Confusión – Test")


# d) Sensibilidad al tamaño de entrenamiento

In [12]:
# Se realiza el nuevo Split entre Train y Test
X_train60, X_test40, y_train60, y_test40 = train_test_split(
    X, y, 
    test_size=0.4, 
    random_state=SEED, 
    stratify=y
)

# Proporción de clases en train y test de HeartDisease
print("Proporción de clases en Nuevo Train:")
print(pd.Series(y_train60).value_counts(normalize=True))
print("\nProporción de clases en Nuevo Test:")
print(pd.Series(y_test40).value_counts(normalize=True))

Proporción de clases en Nuevo Train:
1    0.552727
0    0.447273
Name: proportion, dtype: float64

Proporción de clases en Nuevo Test:
1    0.554348
0    0.445652
Name: proportion, dtype: float64


In [13]:
# Se calcula la Media y Desviación Estándar de cada característica en el conjunto de entrenamiento y Test

print("Conjunto de Nuevo Entrenamiento:")
for i in posición_característica:
    print(f"Característica {feature_names[i]} - Media: {np.mean(X_train60[:, i]):.2f}, Desviación Estándar: {np.std(X_train60[:, i]):.2f}")

print("\nConjunto de Nuevo Test:")
for i in posición_característica:
    print(f"Característica {feature_names[i]} - Media: {np.mean(X_test40[:, i]):.2f}, Desviación Estándar: {np.std(X_test40[:, i]):.2f}")

Conjunto de Nuevo Entrenamiento:
Característica Age - Media: 53.58, Desviación Estándar: 9.30
Característica MaxHR - Media: 136.82, Desviación Estándar: 24.28

Conjunto de Nuevo Test:
Característica Age - Media: 53.41, Desviación Estándar: 9.61
Característica MaxHR - Media: 136.79, Desviación Estándar: 27.10


In [14]:
# Se realiza un histograma de la caracteristica "Cholesterol" de Train y Test
fig = make_subplots(rows=2, cols=1, subplot_titles=("Histograma de Cholesterol en Nuevo Train", "Histograma de Cholesterol en Nuevo Test"))

fig.add_trace(go.Histogram(x=X_train60[:, 4], name="Train"), row=1, col=1)
fig.add_trace(go.Histogram(x=X_test40[:, 4], name="Test", marker_color="red"), row=2, col=1)
fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig.update_yaxes(title_text="Frecuencia", row=2, col=1)

fig.update_layout(height=600, showlegend=False)
fig.show()

## Nuevo Baseline

In [15]:
# Nuevo Train del modelo de Regresión Logística
lr_60 = LogisticRegression(random_state=SEED, max_iter=1000)
lr_60.fit(X_train60, y_train60)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",7
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multic

In [16]:
# Predicciones 2
y_pred_train60 = lr_60.predict(X_train60)
y_pred_test40 = lr_60.predict(X_test40)

In [17]:
metrics_train = metricas_M(y_train60, y_pred_train60, "TRAIN")
metrics_test  = metricas_M(y_test40,  y_pred_test40,  "TEST")

# Tabla comparativa Train vs Test
tabla = pd.DataFrame({"Train": metrics_train, "Test": metrics_test}).T
print("\n── Tabla comparativa ─────────────────────────")
print(tabla.round(4).to_string())


── Métricas TRAIN ──────────────────────────
  Accuracy    : 0.8709
  Precision   : 0.8722
  Recall (TPR): 0.8980
  Specificity : 0.8374
  F1-Score    : 0.8849
  FPR         : 0.1626
  FNR         : 0.1020

── Métricas TEST ──────────────────────────
  Accuracy    : 0.8723
  Precision   : 0.8792
  Recall (TPR): 0.8922
  Specificity : 0.8476
  F1-Score    : 0.8856
  FPR         : 0.1524
  FNR         : 0.1078

── Tabla comparativa ─────────────────────────
       Accuracy  Precision  Recall  Specificity      F1     FPR     FNR
Train    0.8709     0.8722  0.8980       0.8374  0.8849  0.1626  0.1020
Test     0.8723     0.8792  0.8922       0.8476  0.8856  0.1524  0.1078


In [18]:
plot_cm_plotly(y_train60, y_pred_train60, "Matriz de Confusión – Train")
plot_cm_plotly(y_test40,  y_pred_test40,  "Matriz de Confusión – Test")

# e) Curva ROC y AUC

Se calcula la curva ROC, la cual es la curva entre el TPR vs FPR, mientras que el AUC es el área bajo la curva ROC 

$AUC = \displaystyle \int_{0}^{1}ROC(TPR) dTPR$

In [19]:
from sklearn.metrics import roc_curve, roc_auc_score

def plot_roc_curves(splits_dict: dict, titulo: str = "Curva ROC") -> dict:
    """
    Grafica curvas ROC para múltiples modelos/divisiones y reporta AUC.

    Parámetros
    ----------
    splits_dict : dict
        Diccionario con estructura:
        {
            "nombre_curva": {
                "modelo": modelo_sklearn,
                "X": array_features,
                "y": array_labels,
                "color": "#hexcolor"  (opcional)
            },
            ...
        }
    titulo : str
        Título del gráfico.

    Retorna
    -------
    dict : {nombre_curva: auc_value}
    """
    # Paleta de colores por defecto si no se especifica
    paleta_default = [
        "#1f77b4", "#aec7e8", "#d62728", "#f5a9a9",
        "#2ca02c", "#98df8a", "#ff7f0e", "#ffbb78",
    ]

    fig = go.Figure()
    auc_resultados = {}

    for idx, (nombre, contenido) in enumerate(splits_dict.items()):
        modelo  = contenido["modelo"]
        X_split = contenido["X"]
        y_split = contenido["y"]
        color   = contenido.get("color", paleta_default[idx % len(paleta_default)])

        proba = modelo.predict_proba(X_split)[:, 1]
        fpr, tpr, _ = roc_curve(y_split, proba)
        auc = roc_auc_score(y_split, proba)
        auc_resultados[nombre] = auc

        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            mode="lines",
            name=f"{nombre}",
            line=dict(color=color, width=2),
        ))

    # Línea de clasificador aleatorio
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        mode="lines",
        name="Aleatorio (AUC=0.50)",
        line=dict(color="gray", dash="dash", width=1),
    ))

    fig.update_layout(
        title=titulo,
        xaxis_title="FPR",
        yaxis_title="TPR",
        xaxis=dict(range=[0, 1]),
        yaxis=dict(range=[0, 1]),
        legend=dict(x=0.55, y=0.08, bgcolor="rgba(255,255,255,0.8)"),
        width=680, height=520,
    )
    fig.show()

    # Reporte en consola
    print(f"\n── AUC – {titulo} {'─'*30}")
    for nombre, auc in auc_resultados.items():
        print(f"  {nombre:<25}: {auc:.4f}")

    return auc_resultados

In [20]:
splits_dict1 = {
    "Train": {"modelo": lr_80, "X": X_train_80, "y": y_train_80, "color": "#1f77b4"},
    "Test" : {"modelo": lr_80, "X": X_test_20,  "y": y_test_20,  "color": "#aec7e8"},
}


auc_resultados1 = plot_roc_curves(
    splits_dict1,
    titulo="Curva ROC – Regresión Logística Baseline 80/20"
)


── AUC – Curva ROC – Regresión Logística Baseline 80/20 ──────────────────────────────
  Train                    : 0.9333
  Test                     : 0.9304


In [21]:
splits_dict2 = {
    "Train": {"modelo": lr_60, "X": X_train60, "y": y_train60, "color": "#d62728"},
    "Test" : {"modelo": lr_60, "X": X_test40,  "y": y_test40,  "color": "#f5a9a9"},
}
auc_resultados2 = plot_roc_curves(
    splits_dict2,
    titulo="Curva ROC – Regresión Logística Baseline 60/40"
)

# Brechas train-test
gap_80 = auc_resultados1["Train"] - auc_resultados1["Test"]
gap_60 = auc_resultados2["Train"] - auc_resultados2["Test"]
print(f"\n  Brecha Train-Test 80/20: {gap_80:.4f}")
print(f"  Brecha Train-Test 60/40: {gap_60:.4f}")



── AUC – Curva ROC – Regresión Logística Baseline 60/40 ──────────────────────────────
  Train                    : 0.9313
  Test                     : 0.9328

  Brecha Train-Test 80/20: 0.0029
  Brecha Train-Test 60/40: -0.0015


In [64]:
def plot_roc_curves(splits_dict: dict, titulo: str = "Curva ROC") -> dict:
    """
    Grafica curvas ROC estilizadas al estilo bancario para múltiples modelos/divisiones.

    Parámetros
    ----------
    splits_dict : dict
        {
            "nombre_curva": {
                "modelo": modelo_sklearn,
                "X": array_features,
                "y": array_labels,
                "color": "#hexcolor"  (opcional)
            }, ...
        }
    titulo : str
        Título del gráfico.

    Retorna
    -------
    dict : {nombre_curva: auc_value}
    """
    paleta_default = [
        "#1f77b4", "#aec7e8", "#d62728", "#f5a9a9",
        "#2ca02c", "#98df8a", "#ff7f0e", "#ffbb78",
    ]

    fig = go.Figure()
    auc_resultados = {}

    for idx, (nombre, contenido) in enumerate(splits_dict.items()):
        modelo  = contenido["modelo"]
        X_split = contenido["X"]
        y_split = contenido["y"]
        color   = contenido.get("color", paleta_default[idx % len(paleta_default)])

        proba = modelo.predict_proba(X_split)[:, 1]
        fpr, tpr, _ = roc_curve(y_split, proba)
        auc = roc_auc_score(y_split, proba)
        auc_resultados[nombre] = auc

        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            mode="lines",
            name=f"{nombre}",
            line=dict(color=color, width=2),
        ))

    # ── Línea AUC = 0.5 ───────────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        mode="lines",
        name="AUC = 0.5",
        line=dict(color="gray", dash="dash", width=1.5),
    ))

    # ── Línea de referencia TPR = 1 ───────────────────────────────────────────
    fig.add_shape(
        type="line",
        x0=0, x1=1, y0=1, y1=1,
        line=dict(color="gray", dash="dash", width=1.5),
    )
    fig.add_annotation(
        x=0.01, y=1.03,
        text="TPR = 1",
        showarrow=False,
        font=dict(size=12, color="#444444"),
        xanchor="left",
    )

    # ── Layout estilo banco ───────────────────────────────────────────────────
    fig.update_layout(
        title=dict(
            text=titulo,
            font=dict(size=18, color="#333333", family="Times New Roman", weight="bold"),
            x=0.4,           # ← centrado
            xanchor="center", # ← ancla al centro
        ),
        xaxis=dict(
            title=dict(
                text="<b>False Positive Rate</b>",
                font=dict(size=13, color="#333333", family="Times New Roman", weight="bold"),
            ),
            tickvals=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
            tickformat=".1f",
            showgrid=False,
            zeroline=False,
            range=[0, 1],
            showline=True,        # ← línea del eje X
            linecolor="#333333",  # ← color de la línea
            linewidth=1.5,        # ← grosor
        ),
        yaxis=dict(
            title=dict(
                text="<b>True Positive Rate</b>",
                font=dict(size=13, color="#333333", family="Times New Roman", weight="bold"),
            ),
            showticklabels=False,   # sin marcas en eje Y
            showgrid=False,
            zeroline=False,
            range=[0, 1.08],        # espacio para la anotación TPR=1
        ),
        legend=dict(
            x=0.55, y=0.12,
            bgcolor="rgba(240,240,240,0.85)",
            bordercolor="#cccccc",
            borderwidth=1,
            font=dict(size=11),
            title=dict(text=""),
        ),
        plot_bgcolor="white",
        paper_bgcolor="white",
        width=680, height=520,
        margin=dict(t=70, l=60, r=40, b=60),
    )
    fig.show()

    # ── Reporte consola ───────────────────────────────────────────────────────
    print(f"\n── AUC – {titulo} {'─'*30}")
    for nombre, auc in auc_resultados.items():
        print(f"  {nombre:<25}: {auc:.4f}")

    return auc_resultados

In [65]:
splits_dict3 = {
    "Train": {"modelo": lr_80, "X": X_train_80, "y": y_train_80, "color": "#08306b"},  # azul oscuro
    "Test" : {"modelo": lr_80, "X": X_test_20,  "y": y_test_20,  "color": "#6baed6"},  # azul claro
}

auc_resultados3 = plot_roc_curves(
    splits_dict3,
    titulo="Curva ROC – Regresión Logística Baseline (80/20 vs 60/40)"
)


── AUC – Curva ROC – Regresión Logística Baseline (80/20 vs 60/40) ──────────────────────────────
  Train                    : 0.9333
  Test                     : 0.9304


In [66]:
splits_dict4 = {
    "Train": {"modelo": lr_60, "X": X_train60, "y": y_train60, "color": "#67000d"},  # rojo oscuro
    "Test" : {"modelo": lr_60, "X": X_test40,  "y": y_test40,  "color": "#fc8d59"},  # rojo claro
}

auc_resultados4 = plot_roc_curves(
    splits_dict4,
    titulo="Curva ROC – Regresión Logística Baseline (80/20 vs 60/40)"
)

gap_80 = auc_resultados3["Train"] - auc_resultados3["Test"]
gap_60 = auc_resultados4["Train"] - auc_resultados4["Test"]
print(f"\n  Brecha Train-Test 80/20: {gap_80:.4f}")
print(f"  Brecha Train-Test 60/40: {gap_60:.4f}")



── AUC – Curva ROC – Regresión Logística Baseline (80/20 vs 60/40) ──────────────────────────────
  Train                    : 0.9313
  Test                     : 0.9328

  Brecha Train-Test 80/20: 0.0029
  Brecha Train-Test 60/40: -0.0015


# g) GridSearch CV

Se realizará Validación Crusada con K=5 dentro del conjunto de entrenamiento

In [52]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

# Grilla de hiperparámetros
param_grid = [
    # L1 y L2: compatibles con solver='saga'
    {
        "penalty"     : ["l1", "l2"],
        "C"           : [0.1, 1, 10],
        "class_weight": ["balanced", None, {0: 1, 1: 5}],
        "solver"      : ["saga"],
        "l1_ratio"    : [1.0],          # ignorado por L1/L2, pero requerido por saga
        "max_iter"    : [2000],
    },
    # ElasticNet: solo compatible con solver='saga' y requiere l1_ratio
    {
        "penalty"     : ["elasticnet"],
        "C"           : [0.1, 1, 10],
        "class_weight": ["balanced", None, {0: 1, 1: 5}],
        "solver"      : ["saga"],
        "l1_ratio"    : [0.5],          # mezcla 50% L1 - 50% L2
        "max_iter"    : [2000],
    },
]

# Métricas a evaluar simultáneamente
scoring = {
    "AUC"     : "roc_auc",          # ← string directo, más robusto
    "Accuracy": "accuracy",
    "F1"      : "f1",
}

grid_search = GridSearchCV(
    estimator  = LogisticRegression(random_state=SEED),
    param_grid = param_grid,
    scoring    = scoring,
    refit      = "AUC",
    cv         = 5,
    n_jobs     = -1,
    verbose    = 1,
)

grid_search.fit(X_train_80, y_train_80)


Fitting 5 folds for each of 27 candidates, totalling 135 fits


/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemest

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...andom_state=7)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'C': [0.1, 1, ...], 'class_weight': ['balanced', None, ...], 'l1_ratio': [1.0], 'max_iter': [2000], ...}, {'C': [0.1, 1, ...], 'class_weight': ['balanced', None, ...], 'l1_ratio': [0.5], 'max_iter': [2000], ...}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'AUC': 'roc_auc', 'Accuracy': 'accuracy', 'F1': 'f1'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'AUC'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold t

Se realiza una tabla con los resultados

In [53]:
# Tabla de resultados 
results = pd.DataFrame(grid_search.cv_results_)

cols_param   = ["param_penalty", "param_C", "param_class_weight"]
cols_metrics = ["mean_test_AUC", "mean_test_Accuracy", "mean_test_F1"]

tabla_grid = (
    results[cols_param + cols_metrics]
    .rename(columns={
        "param_penalty"     : "Penalty",
        "param_C"           : "C",
        "param_class_weight": "Class Weight",
        "mean_test_AUC"     : "AUC",
        "mean_test_Accuracy": "Accuracy",
        "mean_test_F1"      : "F1-Score",
    })
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)

tabla_grid[["AUC", "Accuracy", "F1-Score"]] = tabla_grid[
    ["AUC", "Accuracy", "F1-Score"]
].round(4)

print(tabla_grid.to_string(index=False))

   Penalty    C Class Weight    AUC  Accuracy  F1-Score
elasticnet 10.0 {0: 1, 1: 5} 0.9052    0.7411    0.8056
        l2 10.0 {0: 1, 1: 5} 0.9052    0.7411    0.8056
        l1 10.0 {0: 1, 1: 5} 0.9051    0.7411    0.8056
        l2  1.0 {0: 1, 1: 5} 0.9051    0.7411    0.8056
elasticnet  1.0 {0: 1, 1: 5} 0.9048    0.7411    0.8056
        l1  1.0 {0: 1, 1: 5} 0.9046    0.7411    0.8060
        l2  0.1 {0: 1, 1: 5} 0.9044    0.7384    0.8053
elasticnet  0.1 {0: 1, 1: 5} 0.9024    0.7261    0.7979
        l1  0.1 {0: 1, 1: 5} 0.9002    0.7193    0.7943
elasticnet 10.0     balanced 0.8921    0.8297    0.8436
        l2  1.0     balanced 0.8921    0.8297    0.8436
        l1 10.0     balanced 0.8920    0.8297    0.8436
        l2 10.0     balanced 0.8920    0.8297    0.8436
        l2 10.0         None 0.8920    0.8352    0.8528
elasticnet 10.0         None 0.8919    0.8352    0.8528
        l2  1.0         None 0.8919    0.8338    0.8514
        l1 10.0         None 0.8919    0.8352   

In [32]:
# ── Mejor combinación 
print("\n── Mejor combinación (por AUC) ────────────────────────")
best = grid_search.best_params_
print(f"  Penalty     : {best['penalty']}")
print(f"  C           : {best['C']}")
print(f"  Class Weight: {best['class_weight']}")
print(f"  AUC CV      : {grid_search.best_score_:.4f}")


── Mejor combinación (por AUC) ────────────────────────
  Penalty     : l2
  C           : 10
  Class Weight: {0: 1, 1: 5}
  AUC CV      : 0.9052


# h) Magnitud de los coeficientes

In [33]:
# Todas las combinaciones de la parte (g)
combinaciones = [
    {"penalty": p, "C": c, "class_weight": cw}
    for p  in ["l1", "l2", "elasticnet"]
    for c  in [0.1, 1, 10]
    for cw in ["balanced", None, {0: 1, 1: 5}]
]

assert len(combinaciones) == 27, "Deben ser exactamente 27 combinaciones"

In [34]:
# Ajuste de los 27 modelos y cálculo de ||β_m||_2
resultados = []
for combo in combinaciones:
    modelo = LogisticRegression(
        penalty      = combo["penalty"],
        C            = combo["C"],
        class_weight = combo["class_weight"],
        solver       = "saga",
        l1_ratio     = 0.5 if combo["penalty"] == "elasticnet" else 1.0,
        max_iter     = 2000,
        random_state = SEED,
    )
    modelo.fit(X_train_80, y_train_80)

    norma = np.linalg.norm(modelo.coef_)

    # Etiqueta legible para class_weight
    if combo["class_weight"] is None:
        cw_label = "None"
    elif combo["class_weight"] == "balanced":
        cw_label = "balanced"
    else:
        cw_label = "{0:1, 1:5}"

    resultados.append({
        "penalty"     : combo["penalty"],
        "C"           : combo["C"],
        "class_weight": cw_label,
        "norma"       : norma,
    })

df_normas = pd.DataFrame(resultados).sort_values(
    ["penalty", "C"], ascending=True
).reset_index(drop=True)

print(df_normas.to_string(index=False))

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, l

   penalty    C class_weight    norma
elasticnet  0.1     balanced 0.974630
elasticnet  0.1         None 0.973217
elasticnet  0.1   {0:1, 1:5} 1.442644
elasticnet  1.0     balanced 1.079815
elasticnet  1.0         None 1.078889
elasticnet  1.0   {0:1, 1:5} 1.554122
elasticnet 10.0     balanced 1.090992
elasticnet 10.0         None 1.090113
elasticnet 10.0   {0:1, 1:5} 1.565736
        l1  0.1     balanced 0.908732
        l1  0.1         None 0.906794
        l1  0.1   {0:1, 1:5} 1.387944
        l1  1.0     balanced 1.072479
        l1  1.0         None 1.071513
        l1  1.0   {0:1, 1:5} 1.548165
        l1 10.0     balanced 1.090247
        l1 10.0         None 1.089364
        l1 10.0   {0:1, 1:5} 1.565142
        l2  0.1     balanced 1.043521
        l2  0.1         None 1.042522
        l2  0.1   {0:1, 1:5} 1.499462
        l2  1.0     balanced 1.087209
        l2  1.0         None 1.086322
        l2  1.0   {0:1, 1:5} 1.560047
        l2 10.0     balanced 1.091737
        l2 1

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [42]:
fig = go.Figure()

for penalty, color in colores_penalty.items():
    subset = df_normas[df_normas["penalty"] == penalty]["norma"]
    fig.add_trace(go.Histogram(
        x            = subset,
        name         = penalty,
        opacity      = 0.85,
        nbinsx       = 8,
    ))

fig.update_layout(
    title   = "Histograma de ||β_m||₂ por tipo de penalty",
    xaxis_title = "||β_m||₂",
    yaxis_title = "Frecuencia",
    barmode = "overlay",
    width   = 680,
    height  = 460,
)

fig.show()

In [44]:
# ── Histograma coloreado por penalty (estilo matching ROC) ────────────────────

colores_penalty = {
    "l1"        : "#1d9e30",   # rojo oscuro
    "l2"        : "#3f00fd",   # rojo medio
    "elasticnet": "#a50000",   # rojo claro / naranja
}

fig = go.Figure()

for penalty, color in colores_penalty.items():
    subset = df_normas[df_normas["penalty"] == penalty]["norma"]
    fig.add_trace(go.Histogram(
        x            = subset,
        name         = penalty,
        marker_color = color,
        opacity      = 0.85,
        nbinsx       = 8,
    ))

fig.update_layout(
    title = dict(
        text    = "Histograma de ||β_m||₂ por tipo de penalty",
        font    = dict(size=14, color="#333333"),
        x       = 0.5,
        xanchor = "center",
    ),
    barmode = "overlay",
    xaxis = dict(
        title = dict(
            text = "||β_m||₂",
            font = dict(size=12, color="#333333"),
        ),
        showgrid  = False,
        showline  = True,
        linecolor = "#333333",
        linewidth = 1.5,
        zeroline  = False,
    ),

    yaxis = dict(
        title = dict(
            text = "Frecuencia",
            font = dict(size=12, color="#333333"),
        ),
        showgrid  = True,
        gridcolor = "rgba(200,200,200,0.3)",
        zeroline  = False,
    ),
    legend = dict(
        title      = dict(text="Penalty"),
        x          = 0.75,
        y          = 0.95,
        bgcolor    = "rgba(240,240,240,0.85)",
        bordercolor= "#cccccc",
        borderwidth= 1,
        font       = dict(size=11),
    ),
    font          = dict(color="#333333"),
    plot_bgcolor  = "rgba(220,230,242,0.4)",   # fondo azul grisáceo suave
    paper_bgcolor = "white",
    width  = 680,
    height = 460,
)

fig.show()

# i) Contribución Relativa de Variables

In [45]:
from sklearn.metrics.pairwise import cosine_similarity

# i) Contribución relativa de variables 

# Vector unitario β̃_m = β̂_m / ||β̂_m||_2 para cada modelo
registros = []
beta_unitarios = {}  # guardamos para análisis posterior

for combo in combinaciones:
    modelo = LogisticRegression(
        penalty      = combo["penalty"],
        C            = combo["C"],
        class_weight = combo["class_weight"],
        solver       = "saga",
        l1_ratio     = 0.5 if combo["penalty"] == "elasticnet" else 1.0,
        max_iter     = 2000,
        random_state = SEED,
    )
    modelo.fit(X_train_80, y_train_80)

    beta_hat  = modelo.coef_.flatten()
    norma     = np.linalg.norm(beta_hat)
    beta_til  = beta_hat / norma          # vector unitario

    # Variable dominante j*(m)
    j_star    = np.argmax(np.abs(beta_til))
    var_dom   = feature_names[j_star]

    # Etiqueta class_weight
    if combo["class_weight"] is None:
        cw_label = "None"
    elif combo["class_weight"] == "balanced":
        cw_label = "balanced"
    else:
        cw_label = "{0:1, 1:5}"

    key = (combo["penalty"], combo["C"], cw_label)
    beta_unitarios[key] = beta_til

    registros.append({
        "penalty"      : combo["penalty"],
        "C"            : combo["C"],
        "class_weight" : cw_label,
        "var_dominante": var_dom,
        "n_nulos"      : int(np.sum(np.abs(beta_til) < 1e-6)),
    })

df_contrib = pd.DataFrame(registros)

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, l

## i) Tabla de variable dominante

In [46]:
print("── (i) Variable dominante por modelo ─────────────────────────────")
print(df_contrib[["penalty","C","class_weight","var_dominante"]].to_string(index=False))

# Frecuencia de cada variable dominante
print("\n── Frecuencia de j*(m) ────────────────────────────────────────────")
print(df_contrib["var_dominante"].value_counts().to_string())

── (i) Variable dominante por modelo ─────────────────────────────
   penalty    C class_weight var_dominante
        l1  0.1     balanced       Oldpeak
        l1  0.1         None       Oldpeak
        l1  0.1   {0:1, 1:5} ST_Slope_Flat
        l1  1.0     balanced ST_Slope_Flat
        l1  1.0         None ST_Slope_Flat
        l1  1.0   {0:1, 1:5} ST_Slope_Flat
        l1 10.0     balanced ST_Slope_Flat
        l1 10.0         None ST_Slope_Flat
        l1 10.0   {0:1, 1:5} ST_Slope_Flat
        l2  0.1     balanced ST_Slope_Flat
        l2  0.1         None ST_Slope_Flat
        l2  0.1   {0:1, 1:5} ST_Slope_Flat
        l2  1.0     balanced ST_Slope_Flat
        l2  1.0         None ST_Slope_Flat
        l2  1.0   {0:1, 1:5} ST_Slope_Flat
        l2 10.0     balanced ST_Slope_Flat
        l2 10.0         None ST_Slope_Flat
        l2 10.0   {0:1, 1:5} ST_Slope_Flat
elasticnet  0.1     balanced       Oldpeak
elasticnet  0.1         None ST_Slope_Flat
elasticnet  0.1   {0:1, 1:5} S

## iii) Histograma agregado de |β̃_m,j| por penalty 

In [47]:
fig = go.Figure()

for penalty in ["l1", "l2", "elasticnet"]:
    # Concatenar todos los |β̃_m,j| de los modelos con ese penalty
    vals = np.concatenate([
        np.abs(v)
        for k, v in beta_unitarios.items()
        if k[0] == penalty
    ])
    fig.add_trace(go.Histogram(
        x      = vals,
        name   = penalty,
        opacity= 0.75,
        nbinsx = 30,
    ))

fig.update_layout(
    title      = "Histograma de |β̃_m,j| por tipo de penalty",
    xaxis_title= "|β̃_m,j|",
    yaxis_title= "Frecuencia",
    barmode    = "overlay",
    width=680, height=460,
)
fig.show()

## iv)  Distancia coseno respecto a la mediana de los β̃_m

In [48]:
# Mediana componente a componente
matriz_betas = np.array(list(beta_unitarios.values()))
mediana_vec  = np.median(matriz_betas, axis=0).reshape(1, -1)

print("\n── (iv) Distancia coseno respecto a la mediana ────────────────────")
for (key, beta_til), reg in zip(beta_unitarios.items(), registros):
    sim  = cosine_similarity(beta_til.reshape(1, -1), mediana_vec)[0][0]
    dist = 1 - sim
    print(f"  {str(key):<45}  dist_coseno={dist:.4f}  n_nulos={reg['n_nulos']}")


── (iv) Distancia coseno respecto a la mediana ────────────────────
  ('l1', 0.1, 'balanced')                        dist_coseno=0.0109  n_nulos=3
  ('l1', 0.1, 'None')                            dist_coseno=0.0105  n_nulos=3
  ('l1', 0.1, '{0:1, 1:5}')                      dist_coseno=0.0118  n_nulos=3
  ('l1', 1, 'balanced')                          dist_coseno=0.0001  n_nulos=1
  ('l1', 1, 'None')                              dist_coseno=0.0001  n_nulos=1
  ('l1', 1, '{0:1, 1:5}')                        dist_coseno=0.0139  n_nulos=0
  ('l1', 10, 'balanced')                         dist_coseno=0.0001  n_nulos=0
  ('l1', 10, 'None')                             dist_coseno=0.0000  n_nulos=0
  ('l1', 10, '{0:1, 1:5}')                       dist_coseno=0.0145  n_nulos=0
  ('l2', 0.1, 'balanced')                        dist_coseno=0.0001  n_nulos=0
  ('l2', 0.1, 'None')                            dist_coseno=0.0000  n_nulos=0
  ('l2', 0.1, '{0:1, 1:5}')                      dist_coseno=0

# k) Comparación de clasificadores

In [54]:
from sklearn.linear_model import Perceptron
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Comparación de clasificadores 

# Mejor configuración de Regresión Logística de j)
lr_best = LogisticRegression(
    penalty      = "l2",
    C            = 10,
    class_weight = {0: 1, 1: 5},
    solver       = "saga",
    l1_ratio     = 1.0,
    max_iter     = 2000,
    random_state = SEED,
)

# Modelos a comparar
modelos = {
    "Logística (mejor)"  : lr_best,
    "Perceptrón"         : Perceptron(random_state=SEED, max_iter=2000),
    "KNN"                : KNeighborsClassifier(),
    "Gaussian Naive Bayes": GaussianNB(),
    "LDA"                : LinearDiscriminantAnalysis(),
}

# Función auxiliar para modelos sin predict_proba (Perceptrón)
def get_scores(modelo, X):
    if hasattr(modelo, "predict_proba"):
        return modelo.predict_proba(X)[:, 1]
    else:
        return modelo.decision_function(X)

# Ajuste y métricas de todos los modelos
resultados_k = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train_80, y_train_80)

    y_pred_tr = modelo.predict(X_train_80)
    y_pred_te = modelo.predict(X_test_20)

    metr_train = metricas_M(y_train_80, y_pred_tr, f"{nombre} – TRAIN")
    metr_test  = metricas_M(y_test_20,  y_pred_te, f"{nombre} – TEST")

    resultados_k[nombre] = {
        "train": metr_train,
        "test" : metr_test,
    }

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=1.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not conv


── Métricas Logística (mejor) – TRAIN ──────────────────────────
  Accuracy    : 0.7589
  Precision   : 0.7071
  Recall (TPR): 0.9631
  Specificity : 0.5061
  F1-Score    : 0.8154
  FPR         : 0.4939
  FNR         : 0.0369

── Métricas Logística (mejor) – TEST ──────────────────────────
  Accuracy    : 0.7391
  Precision   : 0.6824
  Recall (TPR): 0.9902
  Specificity : 0.4268
  F1-Score    : 0.8080
  FPR         : 0.5732
  FNR         : 0.0098

── Métricas Perceptrón – TRAIN ──────────────────────────
  Accuracy    : 0.5722
  Precision   : 0.9035
  Recall (TPR): 0.2537
  Specificity : 0.9665
  F1-Score    : 0.3962
  FPR         : 0.0335
  FNR         : 0.7463

── Métricas Perceptrón – TEST ──────────────────────────
  Accuracy    : 0.5761
  Precision   : 0.8750
  Recall (TPR): 0.2745
  Specificity : 0.9512
  F1-Score    : 0.4179
  FPR         : 0.0488
  FNR         : 0.7255

── Métricas KNN – TRAIN ──────────────────────────
  Accuracy    : 0.8025
  Precision   : 0.8160
  Recall (

Luego, se realiza una tabla de comparación de cada metrica en train y test.

In [55]:
# Tabla comparativa Train 
print("\n── Tabla comparativa – TRAIN ──────────────────────────────────────")
df_train_k = pd.DataFrame(
    {nombre: res["train"] for nombre, res in resultados_k.items()}
).T.round(4)
print(df_train_k.to_string())

# ── Tabla comparativa Test ─────────────────────────────────────────────────────
print("\n── Tabla comparativa – TEST ───────────────────────────────────────")
df_test_k = pd.DataFrame(
    {nombre: res["test"] for nombre, res in resultados_k.items()}
).T.round(4)
print(df_test_k.to_string())

# ── Matrices de confusión ──────────────────────────────────────────────────────
for nombre, modelo in modelos.items():
    plot_cm_plotly(y_train_80, modelo.predict(X_train_80), f"Matriz de Confusión Train – {nombre}")
    plot_cm_plotly(y_test_20,  modelo.predict(X_test_20),  f"Matriz de Confusión Test  – {nombre}")


── Tabla comparativa – TRAIN ──────────────────────────────────────
                      Accuracy  Precision  Recall  Specificity      F1     FPR     FNR
Logística (mejor)       0.7589     0.7071  0.9631       0.5061  0.8154  0.4939  0.0369
Perceptrón              0.5722     0.9035  0.2537       0.9665  0.3962  0.0335  0.7463
KNN                     0.8025     0.8160  0.8300       0.7683  0.8230  0.2317  0.1700
Gaussian Naive Bayes    0.8665     0.8684  0.8941       0.8323  0.8811  0.1677  0.1059
LDA                     0.8774     0.8780  0.9039       0.8445  0.8908  0.1555  0.0961

── Tabla comparativa – TEST ───────────────────────────────────────
                      Accuracy  Precision  Recall  Specificity      F1     FPR     FNR
Logística (mejor)       0.7391     0.6824  0.9902       0.4268  0.8080  0.5732  0.0098
Perceptrón              0.5761     0.8750  0.2745       0.9512  0.4179  0.0488  0.7255
KNN                     0.7120     0.7333  0.7549       0.6585  0.7440  0.3415 

# l) ROC para todos los clasificadores

In [65]:
# l) Curva ROC para todos los clasificadores 

splits_dict_k = {
    "Logística (mejor)" : {
        "modelo": modelos["Logística (mejor)"],
        "X"     : X_test_20,
        "y"     : y_test_20,
        "color" : "#08306b",
    },
    "Perceptrón"        : {
        "modelo": modelos["Perceptrón"],
        "X"     : X_test_20,
        "y"     : y_test_20,
        "color" : "#d62728",
    },
    "KNN"               : {
        "modelo": modelos["KNN"],
        "X"     : X_test_20,
        "y"     : y_test_20,
        "color" : "#2ca02c",
    },
    "Gaussian Naive Bayes": {
        "modelo": modelos["Gaussian Naive Bayes"],
        "X"     : X_test_20,
        "y"     : y_test_20,
        "color" : "#ff7f0e",
    },
    "LDA"               : {
        "modelo": modelos["LDA"],
        "X"     : X_test_20,
        "y"     : y_test_20,
        "color" : "#9467bd",
    },
}

# Necesitamos adaptar plot_roc_curves para manejar
# modelos sin predict_proba (Perceptrón)
def plot_roc_curves_k(splits_dict: dict, titulo: str = "Curva ROC") -> dict:
    fig = go.Figure()
    auc_resultados = {}

    for nombre, contenido in splits_dict.items():
        modelo  = contenido["modelo"]
        X_split = contenido["X"]
        y_split = contenido["y"]

        if hasattr(modelo, "predict_proba"):
            scores = modelo.predict_proba(X_split)[:, 1]
        else:
            scores = modelo.decision_function(X_split)

        fpr, tpr, _ = roc_curve(y_split, scores)
        auc = roc_auc_score(y_split, scores)
        auc_resultados[nombre] = auc

        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            mode="lines",
            name=f"{nombre}",
            line=dict(width=2),
        ))

    # Línea AUC = 0.5
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        mode="lines",
        name="AUC = 0.5",
        line=dict(color="gray", dash="dash", width=1.5),
    ))

    fig.update_layout(
        title      = titulo,
        xaxis_title= "False Positive Rate",
        yaxis_title= "True Positive Rate",
        width=680, height=500,
    )
    fig.update_layout(
    title      = titulo,
    xaxis_title= "False Positive Rate",
    yaxis_title= "True Positive Rate",
    width=680, height=560,   # ← un poco más de altura para dar espacio a la leyenda
    legend=dict(
        orientation="h",     # ← horizontal
        yanchor="top",
        y=-0.2,              # ← debajo del gráfico
        xanchor="center",
        x=0.5,               # ← centrada
    ),
)
    fig.show()

    print(f"\n── AUC – {titulo} {'─'*20}")
    for nombre, auc in sorted(auc_resultados.items(),
                               key=lambda x: x[1], reverse=True):
        print(f"  {nombre:<25}: {auc:.4f}")

    return auc_resultados

auc_k = plot_roc_curves_k(
    splits_dict_k,
    titulo="Curva ROC – Comparación de Clasificadores"
)


── AUC – Curva ROC – Comparación de Clasificadores ────────────────────
  LDA                      : 0.9287
  Logística (mejor)        : 0.9191
  Gaussian Naive Bayes     : 0.9167
  Perceptrón               : 0.7461
  KNN                      : 0.7278


In [60]:
# modelos sin predict_proba (Perceptrón)
def plot_roc_curves_k(splits_dict: dict, titulo: str = "Curva ROC") -> dict:
    fig = go.Figure()
    auc_resultados = {}

    for nombre, contenido in splits_dict.items():
        modelo  = contenido["modelo"]
        X_split = contenido["X"]
        y_split = contenido["y"]
        color   = contenido["color"]

        # Scores: predict_proba si existe, si no decision_function
        if hasattr(modelo, "predict_proba"):
            scores = modelo.predict_proba(X_split)[:, 1]
        else:
            scores = modelo.decision_function(X_split)

        fpr, tpr, _ = roc_curve(y_split, scores)
        auc = roc_auc_score(y_split, scores)
        auc_resultados[nombre] = auc

        fig.add_trace(go.Scatter(
            x=fpr, y=tpr,
            mode="lines",
            name=f"{nombre}  (AUC={auc:.3f})",
            line=dict(color=color, width=2),
        ))

    # Línea AUC = 0.5
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        mode="lines",
        name="AUC = 0.5",
        line=dict(color="gray", dash="dash", width=1.5),
    ))

    # Línea TPR = 1
    fig.add_shape(
        type="line",
        x0=0, x1=1, y0=1, y1=1,
        line=dict(color="gray", dash="dash", width=1.5),
    )
    fig.add_annotation(
        x=0.01, y=1.03,
        text="TPR = 1",
        showarrow=False,
        font=dict(size=12, color="#444444", family="Times New Roman"),
        xanchor="left",
    )

    fig.update_layout(
        title=dict(
            text    = f"<b>{titulo}</b>",
            font    = dict(size=14, color="#333333", family="Times New Roman"),
            x       = 0.5,
            xanchor = "center",
        ),
        xaxis=dict(
            title    = dict(
                text = "<b>False Positive Rate</b>",
                font = dict(size=13, color="#333333", family="Times New Roman"),
            ),
            tickvals  = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
            tickformat= ".1f",
            showgrid  = False,
            zeroline  = False,
            range     = [0, 1],
            showline  = True,
            linecolor = "#333333",
            linewidth = 1.5,
        ),
        yaxis=dict(
            title=dict(
                text = "<b>True Positive Rate</b>",
                font = dict(size=13, color="#333333", family="Times New Roman"),
            ),
            showticklabels = False,
            showgrid       = False,
            zeroline       = False,
            range          = [0, 1.08],
        ),
        legend=dict(
            x=0.55, y=0.12,
            bgcolor    = "rgba(240,240,240,0.85)",
            bordercolor= "#cccccc",
            borderwidth= 1,
            font       = dict(size=11, family="Times New Roman"),
        ),
        font          = dict(family="Times New Roman", color="#333333"),
        plot_bgcolor  = "white",
        paper_bgcolor = "white",
        width=700, height=520,
        margin=dict(t=70, l=60, r=40, b=60),
    )
    fig.show()

    # Reporte AUC
    print(f"\n── AUC – {titulo} {'─'*20}")
    for nombre, auc in sorted(auc_resultados.items(),
                               key=lambda x: x[1], reverse=True):
        print(f"  {nombre:<25}: {auc:.4f}")

    return auc_resultados

auc_k = plot_roc_curves_k(
    splits_dict_k,
    titulo="Curva ROC – Comparación de Clasificadores"
)


── AUC – Curva ROC – Comparación de Clasificadores ────────────────────
  LDA                      : 0.9287
  Logística (mejor)        : 0.9191
  Gaussian Naive Bayes     : 0.9167
  Perceptrón               : 0.7461
  KNN                      : 0.7278


# m) Subconjuntos de variables

In [69]:
# m) Subconjuntos de variables 

# Identificar columnas categóricas y numéricas del dataframe original
# ANTES de aplicar get_dummies
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
num_cols = [c for c in num_cols if c != "HeartDisease"]  # excluir target

print(f"Variables categóricas originales : {cat_cols}")
print(f"Variables numéricas originales   : {num_cols}")

# Crear los tres subconjuntos con dummies aplicados
df_cat = pd.get_dummies(df[cat_cols + ["HeartDisease"]], drop_first=True)
df_num = df[num_cols + ["HeartDisease"]].copy()
df_all = pd.get_dummies(df, drop_first=True)

subconjuntos = {
    "Solo categóricas" : df_cat,
    "Solo numéricas"   : df_num,
    "Todas"            : df_all,
}

# Modelos a evaluar (instancias nuevas para evitar contaminación)
def get_modelos():
    return {
        "Logística (mejor)": LogisticRegression(
            penalty="l2", C=10, class_weight={0:1, 1:5},
            solver="saga", l1_ratio=1.0,
            max_iter=2000, random_state=SEED,
        ),
        "Perceptrón"        : Perceptron(random_state=SEED, max_iter=2000),
        "KNN"               : KNeighborsClassifier(),
        "Gaussian NB"       : GaussianNB(),
        "LDA"               : LinearDiscriminantAnalysis(),
    }

# Entrenamiento y evaluación por subconjunto
resultados_m = []

for nombre_sub, df_sub in subconjuntos.items():
    X_sub = df_sub.drop(columns="HeartDisease").values
    y_sub = df_sub["HeartDisease"].values

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sub, y_sub,
        test_size=0.2, random_state=SEED, stratify=y_sub,
    )

    for nombre_mod, modelo in get_modelos().items():
        modelo.fit(X_tr, y_tr)

        y_pred = modelo.predict(X_te)

        # AUC
        if hasattr(modelo, "predict_proba"):
            scores = modelo.predict_proba(X_te)[:, 1]
        else:
            scores = modelo.decision_function(X_te)
        auc = roc_auc_score(y_te, scores)

        resultados_m.append({
            "Subconjunto": nombre_sub,
            "Modelo"     : nombre_mod,
            "AUC"        : round(auc, 4),
            "Accuracy"   : round(accuracy_score(y_te, y_pred), 4),
            "F1-Score"   : round(f1_score(y_te, y_pred, zero_division=0), 4),
        })

df_m = pd.DataFrame(resultados_m)

/var/folders/c5/g_hqy45s36jc0_wcl6j0lzsm0000gn/T/ipykernel_94354/3584069848.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1

Variables categóricas originales : ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
Variables numéricas originales   : ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']


/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=1.0. penalty is deprecated. Please use l1_ratio o

In [ ]:
# ── Tablas por subconjunto 
for sub in subconjuntos.keys():
    print(f"\n── {sub} ────────────────────────────────────────────")
    print(df_m[df_m["Subconjunto"] == sub]
          .drop(columns="Subconjunto")
          .to_string(index=False))

#  Tabla pivote para comparación directa 
for metrica in ["AUC", "Accuracy", "F1-Score"]:
    pivot = df_m.pivot_table(
        index="Modelo",
        columns="Subconjunto",
        values=metrica,
    )[["Solo categóricas", "Solo numéricas", "Todas"]]
    print(f"\n── {metrica} por modelo y subconjunto ────────────────")
    print(pivot.round(4).to_string())


── Solo categóricas ────────────────────────────────────────────
           Modelo    AUC  Accuracy  F1-Score
Logística (mejor) 0.9172    0.8315    0.8646
       Perceptrón 0.8971    0.8098    0.8309
              KNN 0.9082    0.8315    0.8473
      Gaussian NB 0.8887    0.7989    0.8103
              LDA 0.9143    0.8424    0.8612

── Solo numéricas ────────────────────────────────────────────
           Modelo    AUC  Accuracy  F1-Score
Logística (mejor) 0.8107    0.6304    0.7424
       Perceptrón 0.7434    0.5543    0.3692
              KNN 0.7215    0.7065    0.7404
      Gaussian NB 0.8231    0.7609    0.7843
              LDA 0.8455    0.8043    0.8252

── Todas ────────────────────────────────────────────
           Modelo    AUC  Accuracy  F1-Score
Logística (mejor) 0.9191    0.7391    0.8080
       Perceptrón 0.7461    0.5761    0.4179
              KNN 0.7278    0.7120    0.7440
      Gaussian NB 0.9167    0.8424    0.8543
              LDA 0.9287    0.8478    0.8654

── A